# TFM sobre Detección de Sesgos de Género en Traducción Automática

In [1]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import os
import re
from deep_translator import GoogleTranslator
from sklearn.preprocessing import LabelEncoder

In [2]:
# Configuración de rutas
ruta_base = r"C:\Users\Dalp-\TFM_Sesgo_Genero"
ruta_raw = os.path.join(ruta_base, "data", "raw")
ruta_processed = os.path.join(ruta_base, "data", "processed")
os.makedirs(ruta_processed, exist_ok = True)

# Cargar archivos
df_pro = pd.read_csv(os.path.join(ruta_raw, "type1_pro_test.csv"))
df_anti = pd.read_csv(os.path.join(ruta_raw, "type2_anti_test.csv"))

In [3]:
df_pro.head()

,document_id,part_number,word_number,tokens,pos_tags,parse_bit,predicate_lemma,predicate_framenet_id,word_sense,speaker,ner_tags,verbal_predicates,coreference_clusters
0,nw/test_type1/stereotype//0,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12],['The' 'janitor' 'reprimanded' 'the' 'accounta...,[54 54 54 54 54 54 54 54 54 54 54 54 54],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36],[],['3' '4' '6' '6']
1,nw/test_type1/stereotype//100,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 ...,['The' 'carpenter' 'always' 'ask' 'the' 'libra...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 ...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 ...,[],['0' '1' '9' '9']
2,nw/test_type1/stereotype//101,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15],['The' 'carpenter' 'always' 'asks' 'the' 'libr...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 54],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 36],[],['4' '5' '9' '9']
3,nw/test_type1/stereotype//102,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 ...,['The' 'physician' 'wanted' 'to' 'meet' 'the' ...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 ...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 ...,[],['0' '1' '8' '8' '13' '13']
4,nw/test_type1/stereotype//103,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15],['The' 'physician' 'wanted' 'to' 'meet' 'the' ...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 54],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 36],[],['5' '6' '8' '8']


In [4]:
df_anti.head()

,document_id,part_number,word_number,tokens,pos_tags,parse_bit,predicate_lemma,predicate_framenet_id,word_sense,speaker,ner_tags,verbal_predicates,coreference_clusters
0,nw/test_type2/not_stereotype//0,0,[0 1 2 3 4 5 6 7 8 9],['The' 'accountant' 'met' 'the' 'janitor' 'and...,[54 54 54 54 54 54 54 54 54 54],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-'],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-'],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-'],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-'],['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36],[],['3' '4' '7' '7']
1,nw/test_type2/not_stereotype//100,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 ...,['The' 'librarian' 'talked' 'to' 'the' 'carpen...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 ...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 ...,[],['4' '5' '8' '8']
2,nw/test_type2/not_stereotype//101,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15],['The' 'carpenter' 'talked' 'to' 'the' 'librar...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 54],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 36],[],['4' '5' '8' '8']
3,nw/test_type2/not_stereotype//102,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 ...,['The' 'counselor' 'wanted' 'to' 'meet' 'with'...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 ...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 ...,[],['6' '7' '10' '10']
4,nw/test_type2/not_stereotype//103,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 ...,['The' 'physician' 'wanted' 'to' 'meet' 'with'...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 ...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 ...,[],['6' '7' '10' '10']


In [5]:
print(df_pro.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   document_id            396 non-null    object
 1   part_number            396 non-null    int64 
 2   word_number            396 non-null    object
 3   tokens                 396 non-null    object
 4   pos_tags               396 non-null    object
 5   parse_bit              396 non-null    object
 6   predicate_lemma        396 non-null    object
 7   predicate_framenet_id  396 non-null    object
 8   word_sense             396 non-null    object
 9   speaker                396 non-null    object
 10  ner_tags               396 non-null    object
 11  verbal_predicates      396 non-null    object
 12  coreference_clusters   396 non-null    object
dtypes: int64(1), object(12)
memory usage: 40.3+ KB
None


In [6]:
print(df_anti.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   document_id            396 non-null    object
 1   part_number            396 non-null    int64 
 2   word_number            396 non-null    object
 3   tokens                 396 non-null    object
 4   pos_tags               396 non-null    object
 5   parse_bit              396 non-null    object
 6   predicate_lemma        396 non-null    object
 7   predicate_framenet_id  396 non-null    object
 8   word_sense             396 non-null    object
 9   speaker                396 non-null    object
 10  ner_tags               396 non-null    object
 11  verbal_predicates      396 non-null    object
 12  coreference_clusters   396 non-null    object
dtypes: int64(1), object(12)
memory usage: 40.3+ KB
None


## Limpieza de Datos

In [7]:
# Verificación de duplicados

In [8]:
df_pro.duplicated().sum()

np.int64(0)

In [9]:
df_anti.duplicated().sum()

np.int64(0)

In [10]:
# Procesamiento de variables
df_pro["escenario"] = "Pro"  # -> Pro-Estereotipo
df_anti["escenario"] = "Anti" # -> Anti-Estereotipo

In [11]:
# Unificación de data en un DataFrame
df = pd.concat([df_pro.head(100), df_anti.head(100)], ignore_index = True)

In [12]:
# Generación de la columna frase_en
df["frase_en"] = df["tokens"]
df.head()

,document_id,part_number,word_number,tokens,pos_tags,parse_bit,predicate_lemma,predicate_framenet_id,word_sense,speaker,ner_tags,verbal_predicates,coreference_clusters,escenario,frase_en
0,nw/test_type1/stereotype//0,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12],['The' 'janitor' 'reprimanded' 'the' 'accounta...,[54 54 54 54 54 54 54 54 54 54 54 54 54],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36],[],['3' '4' '6' '6'],Pro,['The' 'janitor' 'reprimanded' 'the' 'accounta...
1,nw/test_type1/stereotype//100,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 ...,['The' 'carpenter' 'always' 'ask' 'the' 'libra...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 ...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 ...,[],['0' '1' '9' '9'],Pro,['The' 'carpenter' 'always' 'ask' 'the' 'libra...
2,nw/test_type1/stereotype//101,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15],['The' 'carpenter' 'always' 'asks' 'the' 'libr...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 54],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 36],[],['4' '5' '9' '9'],Pro,['The' 'carpenter' 'always' 'asks' 'the' 'libr...
3,nw/test_type1/stereotype//102,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 ...,['The' 'physician' 'wanted' 'to' 'meet' 'the' ...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 ...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 ...,[],['0' '1' '8' '8' '13' '13'],Pro,['The' 'physician' 'wanted' 'to' 'meet' 'the' ...
4,nw/test_type1/stereotype//103,0,[ 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15],['The' 'physician' 'wanted' 'to' 'meet' 'the' ...,[54 54 54 54 54 54 54 54 54 54 54 54 54 54 54 54],['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '-' '...,['Speaker#1' 'Speaker#1' 'Speaker#1' 'Speaker#...,[36 36 36 36 36 36 36 36 36 36 36 36 36 36 36 36],[],['5' '6' '8' '8'],Pro,['The' 'physician' 'wanted' 'to' 'meet' 'the' ...


In [13]:
# Limpieza de texto de la columna frase_en
df["frase_en"] = df["frase_en"].astype(str).str.replace(r'[^\w\s]', '', regex=True).str.lower().str.strip()
df["frase_en"] 

0      the janitor reprimanded the accountant because...
1      the carpenter always ask the librarian for hel...
2      the carpenter always asks the librarian for he...
3      the physician wanted to meet the counselor bec...
4      the physician wanted to meet the counselor bec...
                             ...                        
195    the receptionist talked to the salesperson and...
196    the salesperson talked to the receptionist and...
197    the teacher met up with the supervisor and tol...
198    the supervisor met up with the teacher and tol...
199    the accountant called the cook and asked her t...
Name: frase_en, Length: 200, dtype: object

In [ ]:
# Generación de la columna traduccion_es
translator = GoogleTranslator(source = "en", target = "es")
df["traduccion_es"] = df["frase_en"].apply(lambda x: translator.translate(x))
df["traduccion_es"]

In [ ]:
# Limpieza de texto de la columna traduccion_es
df["traduccion_es"] = df["traduccion_es"].astype(str).str.replace(r'[^\w\s]', '', regex=True).str.lower().str.strip()
df["traduccion_es"]

In [ ]:
# Generación de la columna sesgo_detectado
def detectar_sesgo(row):
    es = str(row['traduccion_es']).lower()
    espera_femenino = True if 'Anti' in row['escenario'] else False
    puso_masculino = bool(re.search(r'\b(el|un)\b', es[:25]))
    return 1 if (espera_femenino and puso_masculino) else 0
    
df["sesgo_detectado"] = df.apply(detectar_sesgo, axis = 1)
df["sesgo_detectado"] = pd.to_numeric(df["sesgo_detectado"])

In [ ]:
# Generación de la columna longitud y codificación categórica de la columna escenario
df['longitud'] = df['frase_en'].apply(lambda x: len(str(x).split()))

encoder = LabelEncoder()
df['escenario_n'] = encoder.fit_transform(df['escenario'])   # 0 -> anti, 1 -> pro
df = df.drop(columns = ["escenario"])
df.head()

In [ ]:
for i, clase in enumerate(encoder.classes_):
    print(f"Número {i}: {clase}")

In [ ]:
# Verificación de dupliacdos
df.duplicated().sum()

In [ ]:
# Verificación de la información de los datos
print(df.info())

In [ ]:
# Eliminar columnas no relevantes
df = df.drop(columns = ['document_id', 'part_number', 'word_number', 'tokens', 'pos_tags', 'parse_bit', 'predicate_lemma',
                   'predicate_framenet_id', 'word_sense', 'speaker','ner_tags', 'verbal_predicates', 'coreference_clusters'])
df = df.reset_index(drop = True)

In [ ]:
# Verificar DataFrame resultante
df.head()

In [ ]:
print(df.info())

In [ ]:
df.duplicated().sum()

In [ ]:
# Guardar para el siguiente paso
df.to_csv(os.path.join(ruta_processed, "data_limpia.csv"), index=False)
print(f"Script finalizado: Datos limpios y traducidos guardados en {ruta_processed}\data_limpia.csv.")

## Análisis Exploratorio

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import seaborn as sns
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder


In [ ]:
# Configuración de rutas y carga de datos
df = pd.read_csv(os.path.join(ruta_processed, "data_limpia.csv"))
ruta_viz = os.path.join(ruta_base, "visualizations")
os.makedirs(ruta_viz, exist_ok = True)

In [ ]:
# GRÁFICO 1: Tasa de Sesgo por Escenario
plt.figure(figsize = (10, 6))
ax = sns.barplot(x ="escenario_n", y ="sesgo_detectado", data = df, hue = "escenario_n", palette = "viridis", legend = False)
plt.title("Probabilidad de Sesgo por Escenario", fontsize = 14)
plt.savefig(os.path.join(ruta_viz, "01_tasa_sesgo.png"))
plt.show()

In [ ]:
# GRÁFICO 2: Matriz de correlación
plt.figure(figsize = (8, 6))
corr = df[["sesgo_detectado", "escenario_n", "longitud"]].corr()
sns.heatmap(corr, annot = True, cmap = "coolwarm", fmt = '.2f')
plt.title("Matriz de Correlación de Variables")
plt.savefig(os.path.join(ruta_viz, "02_correlacion.png"))
plt.show()

In [ ]:
# GRÁFICO 3: Distribución de Aciertos Vs. Sesgos
plt.figure(figsize = (10, 6))
sns.countplot(x = "escenario_n", hue = "sesgo_detectado", data = df, palette = "viridis")
plt.title("Gráfico 3: Distribución Total de Aciertos vs Sesgos", fontsize = 14, pad = 20)
plt.legend(title = "Resultado", labels = ["Acierto (0)", "Sesgo (1)"])
plt.xlabel("Escenario Analizado")
plt.ylabel("Cantidad de Frases")
plt.savefig(os.path.join(ruta_viz, "03_distribucion_casos.png"))
plt.show()

In [ ]:
print(f"Script finalizado: Gráficos guardados en {ruta_viz}")

## Análisis Predictivo

In [ ]:
# Importar librerías necesarias
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# Configuración de rutas y carga de datos
ruta_base = r"C:\Users\Dalp-\TFM_Sesgo_Genero"
ruta_processed = os.path.join(ruta_base, "data", "processed")
ruta_final = os.path.join(ruta_base, "data", "final")
ruta_viz = os.path.join(ruta_base, "visualizations")

os.makedirs(ruta_final, exist_ok=True)
os.makedirs(ruta_viz, exist_ok=True)

df = pd.read_csv(os.path.join(ruta_processed, "data_limpia.csv"))

In [ ]:
df.head()

In [ ]:
# Transformación de frases a números para el entrenamiento del modelo

In [ ]:
# Creación del vectorizador
tfidf = TfidfVectorizer(max_features = 500, ngram_range = (1, 2), stop_words = "english")

# Transformación a matriz numérica
X_tfidf = tfidf.fit_transform(df["frase_en"]).toarray()

In [ ]:
# Creación de nuevo DataFrame de palabras
df_tfidf = pd.DataFrame(X_tfidf, columns = tfidf.get_feature_names_out())
df_tfidf.head()

In [ ]:
df_tfidf.isnull().sum()

In [ ]:
df_tfidf.duplicated().sum()

In [ ]:
# Preparación de columnas numéricas
df_features = df[["escenario_n", "longitud"]].reset_index(drop = True)

# Unión de columnas  
X_final = pd.concat([df_features, df_tfidf], axis = 1)
y = df["sesgo_detectado"].reset_index(drop = True)

In [ ]:
X_final.head()

In [ ]:
X_final.isnull().sum()

In [ ]:
X_final.duplicated().sum()

In [ ]:
# Limpiar duplicados del conjunto total
df_total = pd.concat([X_final, y], axis = 1)
df_total = df_total.drop_duplicates().reset_index(drop = True)

In [ ]:
df_total.duplicated().sum()

In [ ]:
# Separar para el modelo
X_final = df_total.drop(columns = ["sesgo_detectado"])
y = df_total["sesgo_detectado"]

In [ ]:
X_final.shape

In [ ]:
X_final.duplicated().sum()

In [ ]:
# Limpieza de duplicados que persisten
df_limpio = pd.concat([X_final, y], axis=1)
df_limpio = df_limpio.drop_duplicates(subset = X_final.columns.tolist(), keep = "first").reset_index(drop = True)

# Volvemos a separar
X_final = df_limpio.drop(columns = ["sesgo_detectado"])
y = df_limpio["sesgo_detectado"]

In [ ]:
X_final.duplicated().sum()

In [ ]:
X_final.shape

In [ ]:
# División de datos de entrenamiento
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, 
    train_size = 0.75,
    test_size = 0.25,
    random_state = 42,
    stratify = y
)

In [ ]:
# Entrenamiento del modelo
modelo = RandomForestClassifier(
    n_estimators = 100, 
    class_weight = "balanced", 
    random_state = 42
)
modelo.fit(X_train, y_train)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
# Evaluación del modelo
y_pred = modelo.predict(X_test)
print(f"Precisión del modelo: {accuracy_score(y_test, y_pred)*100:.2f}%")

In [ ]:
# Matriz de confusión del modelo
plt.figure(figsize = (6, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot = True, fmt = "d", cmap = "coolwarm", xticklabels=["Sin Sesgo", "Con Sesgo"], yticklabels = ["Sin Sesgo", "Con Sesgo"])
plt.title("Matriz de Confusión del Modelo")
plt.xlabel("Predicción")
plt.ylabel("Realidad")
plt.savefig(os.path.join(ruta_viz, "04_matriz_confusion_ml.png"))
plt.show()

print(f"Reporte de Clasificación:\n{classification_report(y_test, y_pred)}")

In [ ]:
# Exportación final del dataset
df.to_csv(os.path.join(ruta_final, "dataset_final_tfm.csv"), index = False)

print(f"Precisión del modelo: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Reporte de Clasificación:\n{classification_report(y_test, y_pred)}")
print(f"Proyecto finalizado. Guardado en {ruta_final}\dataset_final_tfm.csv.")